# Sentiment Analysis using Gemini, Llama3, and OpenAI
## EMC Comments Analysis
### 001 - Data Preparation (Cleansing, Duplicates, NULLs, Prompt Injection Tests)

Read in curated comments from 1 to N datafiles, supplied by CARA extract.

Presentation provided by Power BI.

### TODO
+ Are verbotten phrases being removed?
+ Added JSON_ERROR to try/catch
+ Handle prompt engineering with library, use prompt templates.
+ https://towardsdatascience.com/document-topic-extraction-with-large-language-models-llm-and-the-latent-dirichlet-allocation-e4697e4dae87

### Version History
+ v0.1 - General access, no cleansing of data, df.apply() for OpenAI API call.  Gaps in data output.
+ v0.2 - Same dataset (Sonya Sachedeva), robust cleansing, lemmatizing, and stemming.  Summary of summary for token limit solved.
+ v0.3 - Added prompt defense, PII defense, df.apply() with defensive method, dropping lemmatizing/stemming.  Added libraries such as commonregex, spacy, and transformers.
+ v0.4 - Broke into data processing versus data prepping functions.
+ v0.5 - Moved core code into "main" to support multi-processing in future, backed up original data to 'Letter Text_ORIGINAL', explored multi-processing and GPU utilization.
+ v0.6 - New dataset to process direct from CARA extract.  Sonya Sachedeva's data inputs processed with v0.5 which has been tagged.
+ v0.7 - Added embeddings, added read of coded comments and save to binary file.

In [ ]:
# -*- coding: utf-8 -*-

## Environment Check / Validation

In [1]:
###########################################
#- Minimal imports to start
###########################################
try:
    import sys
    import subprocess
    import importlib.util
    import atexit
except ImportError as e:
    print("There was a problem importing the most basic libraries necessary for this code.")
    print(repr(e))
    raise SystemExit("Stop right there!")

###########################################
#- Final Exit Routine
###########################################
@atexit.register
def goodbye():
    print("GOODBYE")

###########################################
#- Cloud Environment Setup (Priming)
###########################################
# variables establishing environments
ENV_GCP=0
ENV_AZURE=1
user_input=-1
environments=["GCP", "Azure"]
    
#prompt user for environment before continuing
user_input = 1
while True:
  try:
     if user_input > -1:
         break;
     user_input = int(input("Select the environment you're running: (0) GCP (1) Azure"))     
     if user_input > 1:
         print("Not a valid choice, please try again.")
         continue;
  except ValueError:
     print("Not a valid choice, please try again.")
     continue
  else:
     print(f"Environment selected is: {environments[user_input]}")
     break 
        
############################################
#- Import a custom library, in this case a fairly useful logging framework
############################################
from pathlib import Path
debug_lib_location = Path("../ML-Support")
sys.path.append(str(debug_lib_location))
import debug

libraries=["transformers", "langchain", "openpyxl", "backoff", "spacy", "numba", "spacy-transformers", "tensorflow", "tensorrt", "python-dotenv",
           "alive-progress", "tqdm", "pyspellchecker", "wordcloud", "langchain", "icecream", "numba", 
           "fitz","dataclasses", "commonregex", "transformers", "spacy", "PyMuPDF", "PyPDF2", "pdfminer", 
           "pdfplumber","pdf2image","pytesseract", "usaddress", "bs4", "unidecode", "nltk",
           "xlsxwriter", "cupy" ]    

debug.msg_info(f"Validating environment for the following pip packages: {libraries}")

#load environment for non-generative libraries
try:
    for library in libraries:
      if library == "Pillow":
        spec = importlib.util.find_spec("PIL")
      else:
        spec = importlib.util.find_spec(library)
      if spec is None:
        print("...installing library " + library)
        subprocess.run(["pip", "install" , library, "--quiet"])
      else:
        print("...library " + library + " already installed.")
except (subprocess.CalledProcessError, Exception) as e:
    print("Error: Failed to install required packages, your code might not run properly.")
    print(repr(e))

#load environment specific libraries for generative AI.
try:    
    if environments[user_input]=="GCP":
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-aiplatform", "--quiet"])
      gcp_libraries=["google-generativeai", "google-cloud-secret-manager"]
      for library in gcp_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    
        from google.cloud import aiplatform
        import vertexai.preview
        from google.cloud import secretmanager
        import google.generativeai as genai        
        
        
    elif environments[user_input]=="Azure":
      azure_libraries=["openai", ]
      for library in azure_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    else:
        print("There was a problem processing your request.  Only numeric input of 0 or 1 is allowed.")
        print("Continued operations is not possible without the proper installed tools.")
        raise SystemExit("Stop right there!")
except Exception as e:
    print("There was a problem processing library installs for Generative AI libraries")
    print(repr(e))
    raise SystemExit("Stop right there!")

debug.msg_debug("...dynamic environment installs complete.")

[2024-11-18 22:26:48 UTC]    INFO: Validating environment for the following pip packages: ['transformers', 'langchain', 'openpyxl', 'backoff', 'spacy', 'numba', 'spacy-transformers', 'tensorflow', 'tensorrt', 'python-dotenv', 'alive-progress', 'tqdm', 'pyspellchecker', 'wordcloud', 'langchain', 'icecream', 'numba', 'fitz', 'dataclasses', 'commonregex', 'transformers', 'spacy', 'PyMuPDF', 'PyPDF2', 'pdfminer', 'pdfplumber', 'pdf2image', 'pytesseract', 'usaddress', 'bs4', 'unidecode', 'nltk', 'xlsxwriter', 'cupy'] 
...library transformers already installed.
...library langchain already installed.
...library openpyxl already installed.
...library backoff already installed.
...library spacy already installed.
...library numba already installed.
...installing library spacy-transformers
...library tensorflow already installed.
...library tensorrt already installed.
...installing library python-dotenv
...installing library alive-progress
...library tqdm already installed.
...installing librar

## Includes and Libraries

In [ ]:
debug.msg_info("Library imports")    
############################################
# INCLUDES
############################################

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...core libraries.")
import os
import datetime
import gc
import socket
import sys
import getopt
import inspect
import traceback
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
from io import StringIO
import tqdm


import io
import math
import textwrap
import random
import glob
import time
from time import perf_counter
import subprocess
import backoff 
from multiprocessing import Pool

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Function Profiling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import cProfile
import pstats
import io
from pstats import SortKey

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...classic data science libraries.")

#optimization routines
from numba import jit
import numpy as np
import scipy as sp
#from sklearn.linear_model import LinearRegression

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...application specific libraries.")
import math
from base64 import b64decode
from IPython.display import Image
import requests
from bs4 import BeautifulSoup                 #used to parse the text
from wordcloud import WordCloud, STOPWORDS    #custom library specifically designed to make word clouds
from spellchecker import SpellChecker
import fitz
#to handle strange characters
from unidecode import unidecode 
from dotenv import load_dotenv

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...graphics.")
#import PIL
from PIL import Image
import PIL.ImageOps
import matplotlib as matplt
import matplotlib.pyplot as plt

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# progress bar
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...progress bars.")
from alive_progress import alive_bar
#from alive_progress.styles import showtime, Show
from tqdm.notebook import trange, tqdm
#from tqdm import trange, tqdm

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PII libraries (regular expressions)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...regular expressions for PII and transformers for prompt injection defense.")
from commonregex import CommonRegex
from commonregex import email
from commonregex import time
from commonregex import credit_card
from commonregex import ip
from commonregex import ipv6
from commonregex import link
from commonregex import phone
from commonregex import street_address
from commonregex import btc_address

debug.msg_debug("...spacy (pii defense).")
import spacy
from spacy.language import Language
from spacy.tokens import Doc
from spacy.matcher import Matcher

import usaddress

debug.msg_debug("...hugging face model support.")
#injection defense
from transformers import pipeline

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Tensorflow AI/ML libraries (seek to use GPU's)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#load first
try:
    debug.msg_debug("...TensorRT")    
    import tensorrt
    assert tensorrt.Builder(tensorrt.Logger())
except ImportError as ie:
    debug.msg_warning("Failed to import tensorrt, this might be a problem if trying for enhanced processing.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    #load second
    debug.msg_debug("...TensorFlow")        
    import tensorflow as tf
except ImportError as ie:
    debug.msg_warning("Failed to import tensorflow, might not have a GPU or the proper environment loaded")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...CUDF")    
    import cudf
except ImportError as ie:
    debug.msg_warning("Failed to import cudf, likely don't have a GPU")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...Torch")    
    import torch
except ImportError as ie:
    debug.msg_warning("Failed to import torch, likely don't have a GPU or access to that library.")
    debug.msg_warning(f"...{repr(ie)}")
    pass


import pandas as pd
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- NLTK required resources
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...natural language processing.")
import nltk
from nltk.stem import PorterStemmer  # A word stemmer based on the Porter stemming algorithm.  Porter, M. "An algorithm for suffix stripping." Program 14.3 (1980): 130-137.
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tree import tree
#from nltk.book import *
from nltk import FreqDist
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords    

nltk.download('punkt')
nltk.download("words")
nltk.download("stopwords")
#nltk.download('averaged_perceptron_tagger')      #looks like you have to download select neural layers for specific functions, head to read the erorr output to learn this.


## Functions

In [ ]:
def set_library_configuration() -> None:
    
    ############################################
    #- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
    ############################################
    #pandas set floating point to 4 places to things don't run loose
    debug.msg_info("Setting Pandas and Numpy library options.")    
    pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
    pd.options.display.float_format = '{:,.4f}'.format
    np.set_printoptions(precision=4)

In [ ]:
## Manages exception output.
#  @param   (Exception)             - Exception to expound upon
#  @returns (None)                  - None
def process_exception(inc_exception) -> None:
    print(f"{BOLD_START}(Exception encountered):{BOLD_END} {type(inc_exception).__name__}")
    print(f"Details: {str(inc_exception)}")
    print("Traceback:")
    traceback.print_exc()

In [ ]:
def profile_function(func):
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = SortKey.CUMULATIVE
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper

In [ ]:
## Outputs library version history of effort.
#
#  @returns (None)                  - None
def lib_diagnostics() -> None:

    import pkg_resources
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}") 
    
    package_name_length=40
    package_version_length=20

    # Get installed packages
    the_packages=["cupy", "jupyter-core", "langchain", "langchain-core", "nltk", "numba", "numpy", "pandas", "pydantic", "pyspellchecker", "spacy", "scipy", "scikit-learn", "seaborn", "usaddress", "xarray",]
    the_packages.sort()
    
    installed_dict = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    installed=list(installed_dict.keys())
    installed.sort()
    
    #for package_idx, package_name in enumerate(installed):
    for idx, name in enumerate(installed):
         if name in the_packages:
             installed_version = installed_dict[name]
             print(f"{name:<40}#: {str(pkg_resources.parse_version(installed_version)):<20}")
   
    try:
        print(f"{'TensorFlow version':<40}#: {str(tf.__version__):<20}")
        print(f"{'     gpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('GPU')))}")
        print(f"{'     cpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('CPU')))}")
    except Exception as e:
        pass

    try:
        print(f"{'Torch version':<40}#: {str(torch.__version__):<20}")
        print(f"{'     GPUs available?':<40}#: {torch.cuda.is_available()}")
        print(f"{'     count':<40}#: {torch.cuda.device_count()}")
        print(f"{'     current':<40}#: {torch.cuda.current_device()}")
    except Exception as e:
        pass


    try:
      print(f"{'OpenAI Azure Version':<40}#: {str(the_openai_version):<20}")
    except Exception as e:
      pass

    print(f"{BOLD_START}List Devices{BOLD_END} #########################################")
    try:
      from tensorflow.python.client import device_lib
      print(device_lib.list_local_devices())
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(åe)))

    print(f"{BOLD_START}Devices Counts{BOLD_END} ########################################")
    try:
      print(f"Num GPUs Available: {str(len(tf.config.experimental.list_physical_devices('GPU')))}" )
      print(f"Num CPUs Available: {str(len(tf.config.experimental.list_physical_devices('CPU')))}" )
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    print(f"{BOLD_START}Optional Enablement{BOLD_END} ####################################")
    try:
      gpus = tf.config.experimental.list_physical_devices('GPU')
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    if gpus:
      # Restrict TensorFlow to only use the first GPU
      try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print( str( str(len(gpus)) + " Physical GPUs," + str(len(logical_gpus)) + " Logical GPU") )
      except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(str(repr(e)))
      print("")
        
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}") 
    return

In [ ]:
#def fix_periods(inc_str: str) -> str:
#    resultant = re.sub('\.(?!\s|\d|$)', '. ', inc_str)
#    return resultant

## Read the contents of a text input and remove URL's, extra spaces, carriage returns, etc..
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - All junk removed.
def clean_string(inc_str: str) -> str:
    resumeText = re.sub('httpS+s*', ' ', inc_str)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub(r'\r', '', resumeText)
    resumeText = re.sub(r'\n', '', resumeText)
    resumeText = re.sub(' +', ' ', resumeText) # remove extra whitespace
    resumeText = re.sub(r'\t', ' ', resumeText) #remove tabs
    # Remove any potential SQL injection or script injection attempts
    resumeText = re.sub(r'[<>&\'"()]', '', resumeText)
    
    resumeText.rstrip()
    resumeText.lstrip()
    #resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    #resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+-/:;<=>@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations, keep commas and periods
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+/:;<=>@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations, keep commas and periods
    resumeText = ''.join([i if ord(i) < 128 else ' ' for i in resumeText])

    resumeText=re.sub(r'\W+', ' ', resumeText)
    resumeText=re.sub(' +', ' ', resumeText)       
    # Remove punctuation
    #no_punctuation = (nopunc.translate(str.maketrans('', '', string.punctuation)) for nopunc in lower)
    #resumeText = ''.join(x for x in resumeText if x.isalnum())
    #resumeText = re.sub(r'[^x00-x7f]',r' ', resumeText) 
    #resumeText = re.sub('s+', ' ', resumeText)  # remove extra whitespace
    return resumeText

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_stop_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        for word in wordlist:
            if word.casefold() not in stop_words:
              filtered_list.append(word)
        return str(' '.join(filtered_list))

## Read the contents of a text input modify string for stem words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stem words altered
def clean_stem_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        stemmed_words = [stemmer.stem(word) for word in wordlist]
        return str(' '.join(stemmed_words))

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_lemmatizer_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        lemmatized_words = [lemmatizer.lemmatize(word) for word in wordlist]
        return str(' '.join(lemmatized_words))

   
## Read the contents of a text and completely break it down to the lowest level
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Complete purge of all content for NLP
def cleanse_string(inc_str: str) -> str:
    response=clean_string(inc_str)
    response=clean_text(response)
    response=clean_stop_words(response)
    response=clean_stem_words(response)
    response=clean_lemmatizer_words(response)
    response=word_tokenize(response)
    return response

In [ ]:
## Chop up a string based on chunk size and return an array of strings
#
#  @param (Incoming String to Chop)    - str
#  @param (Chunk Size)                 - int
#  @returns ([])                       - list 
def prompt_injection_split_string(your_string, n) -> []:
    return [your_string[i:i + n] for i in range(0, len(your_string), n)]

In [ ]:
## Interprets model results for project injection attacks.
#
#  @param (Transformer Pipeline)    - pipeline - Mechanism via "transformer" library to read neural layer and execute evaluation on it.
#  @param (Text to Analyze, String) - str      - Actual input to evaluate.
#  @returns ({})                    - dict     - Results of neural processing, dictionary of true/false:% quality response
def prompt_injection_predict(inc_pipe: pipeline, inc_prompt: str) -> {}:
        return {id2label[x['label']]: x['score'] for x in inc_pipe(inc_prompt)}

In [ ]:
## Looks for project injection using various neural layers
#
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def detect_PromptInjection(inc_prompt:str) -> bool:

    resultant=False
    offending_content=[]
    keywords=["ignore", "pretend" ]
    prompt_detected=[]
    #debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    try:
        chunk_size=prompt_defense_model_chunk_size
        chunks=prompt_injection_split_string(inc_prompt, chunk_size)
        for model_idx,model_name in enumerate(the_models):
            #debug.msg_debug(f"......processing model {PROMPT_INJECTION_MODELS[model_idx]}")
            the_models[model_idx].to(device=0)
            for chunk_idx, chunk_value in enumerate(chunks):
                the_answer=prompt_injection_predict(the_models[model_idx],str(chunk_value))
                #if an offending answer is found store it for future analysis.
                if (True in list(the_answer.keys())):
                    offending_content.append(chunk_value)
                prompt_detected.append(the_answer)
            #print(results)

    except Exception as e:
        process_exception(f"ERROR predict prompt injection as follows: {str(e)}")
        prompt_detected.append({False:100.0})

    for status in prompt_detected:
        for the_status in status.keys():
            if (the_status):
                resultant=True

    #debug.msg_debug(f"......evaluating keywords")
    wordlist=word_tokenize(inc_prompt.lower())
    for word in wordlist:
        if word in keywords:
            resultant=True

    #debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")
    return resultant, offending_content

In [ ]:
## Looks for PERSON object identified by spacy and abstracts the input to a constant "NAME"
#  https://www.geeksforgeeks.org/python-named-entity-recognition-ner-using-spacy/
#  @param (Spacy Model for Parsing) - Spacy Model
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def clean_named_entity_recognition(inc_model, inc_text) -> str:

    doc = inc_model(inc_text)
    #nec_labels=["PERSON", "ORG", "DATE", "TIME"]
    nec_labels=["PERSON"]

    text_chunks=[]
    cleansed_text=inc_text
    if len(inc_text) > inc_model.max_length-1:
        chunks=int(round(len(inc_text)/(inc_model.max_length-1)))
        start=0
        end=inc_model.max_length-1
        for idx, chunk in enumerate(chunks):
            start=idx*inc_model.max_length-1
            end=(idx) * (inc_model.max_length -1)
            text_chunks.append(inc_text[start:end])
    else:
        text_chunks.append(inc_text)


    for idx, chunk in enumerate(text_chunks):
        doc=inc_model(chunk)    
        for ent in doc.ents:
            if ent.label_ in nec_labels:
                #debug.msg_warning(f" Encountered NER {ent}")
                try:
                    cleansed_text = re.sub(f"{re.escape(str(ent))}", f"{str(ent.label_)}", cleansed_text)
                except Exception as e:
                    debug.msg_warning(f"clean_named_entity_recognition threw an exception: {repr(e)}")
                    pass  #continue processing regardless, we'll accept loss of some Person identification to keep the code processing
    return(cleansed_text)
    

In [ ]:
## Looks for EMAIL object identified by spacy and abstracts the input to a constant "EMAILADDR"
#  https://spacy.pythonhumanities.com/02_02_matcher.html#:~:text=How%20to%20use%20the%20spaCy%20Matcher%201%206.1.,...%205%206.5.%20Finding%20Quotes%20and%20Speakers%20
#  @param (Spacy Model for Parsing)         - Spacy Model
#  @param (Spacy Matcher for Email Pattern) - Spacy Matcher
#  @param (Text to Analyze, String)         - String - Actual input to evaluate.
#  @returns (String)                        - String - Transformed string abstracting email
def clean_email_spacy(inc_model, inc_matcher, inc_text) -> str:

    text_chunks=[]
    cleansed_text=inc_text
    if len(inc_text) > inc_model.max_length-1:
        chunks=int(round(len(inc_text)/(inc_model.max_length-1)))
        start=0
        end=inc_model.max_length-1
        for idx, chunk in enumerate(chunks):
            start=idx*inc_model.max_length-1
            end=(idx) * (inc_model.max_length -1)
            text_chunks.append(inc_text[start:end])
    else:
        text_chunks.append(inc_text)

    for idx, chunk in enumerate(text_chunks):
        doc=inc_model(chunk)
        matches = inc_matcher(doc)            
        for match in matches:
            try:
                cleansed_text = re.sub(f"{str(doc[match[0]:match[1]])}", f"EMAIL", cleansed_text)
            except Exception as e:
                debug.msg_warning(f"clean_email_spacy threw an exception: {repr(e)}")
                pass  #allow it to continue processing, we have other was of managing email
    
    return(cleansed_text)

In [ ]:
## PII Clean - Looks for a variety of specific criteria to transform PII 
#  Could consider adding data to the domain using this technique: https://github.com/lmeulen/PrivacyFilter/tree/master
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def clean_pii(inc_text:str) -> str:
    #debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    try:
    
        #other PII data transformed with regular expressions
        cleansed_text=inc_text
        cleansed_text=re.sub(email,          "EMAIL-ADDR",     cleansed_text)
        cleansed_text=re.sub(credit_card,    "CREDIT-CARD",    cleansed_text)
        cleansed_text=re.sub(link,           "URL",           cleansed_text)
        cleansed_text=re.sub(ip,             "IP",            cleansed_text)
        cleansed_text=re.sub(ipv6,           "IPV6",          cleansed_text)
        cleansed_text=re.sub(phone,          "PHONE-NUMBER",   cleansed_text)
        #cleansed_text=re.sub(street_address, "STREETADDRESS", cleansed_text)
        #cleansed_text=re.sub(btc_address,    "BTCADDRESS",    cleansed_text)

        cleansed_text=re.sub('^\d{5}(?:[-\s]\d{4})?$',"ZIPCODE",cleansed_text)
        cleansed_text=re.sub(' \d{5}(?:[-\s]\d{4})?'," ZIPCODE2",cleansed_text)
        cleansed_text=re.sub(' .. \d{5}. ',"ZIPCODE3",cleansed_text)
        
   
    except Exception as e:
        process_exception(e)
        
#    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")
    return cleansed_text


In [ ]:
## Calculate how long a string is from a dataframe.
#
#  @param (Text for length calculation, str) - str    - Actual text.
#  @returns (int)                            - int    - Actual length of the string.
def calculate_string_size(inc_string:str) -> int:
    response=int(len(word_tokenize(inc_string)))
    return response

## Calculate the metrics for tokens for the entire dataset
#
#  @param (pandas.DataFrame) - pandas.DataFrame    - Dataframe of all data manipulated thus far.
#  @returns (None)           - Just print statements
def get_metrics(inc_df: pd.DataFrame) -> None:
    mylist = []
    the_columns = SOURCE_COLUMNS_NAME

    for idx, name in enumerate(the_columns):
        try:
            new_df = [ mylist.append(calculate_string_size(the_string)) for the_string in tqdm(inc_df[name]) ]
        except Exception as e:
            process_exception(f"Unable to tokenize, and gather metrics for {name}...investigate, continuing:{str(e)}")
        
        arr = np.array(mylist)
        debug.msg_debug(f"Metrics on tokens for {name}.\n")
        debug.msg_debug(f"..records: {len(arr):>30,}")
        debug.msg_debug(f"......max: {np.max(arr):>30,}")
        debug.msg_debug(f"......avg: {int(np.average(arr)):>30,}")
        debug.msg_debug(f"......min: {np.min(arr):>30,}")
        debug.msg_debug("Analyze the result and ensure you have solid data.")

In [2]:
## Load a dataframe from a data file
#
#  @param    (str)         - str             - Filename to read
#  @param    (str)         - str             - Delimiter to parse the data with
#  @returns (pd.DataFrame) - pd.DataFrame    - Actual DataFrame that encapsulates the data.
def load_dataframe(inc_filename:str, inc_encoding:str) -> pd.DataFrame:
inner_df=pd.DataFrame()
    if os.path.isfile(inc_filename):
      #open the file, read the contents and close the file
      try:
          inner_df = pd.read_csv(inc_filename, delimiter=f"{DELIM}")
      except Exception as e:
          process_exception(e)
          raise SystemExit("Stop right there!")  
    else:
      error_msg=f"File not found.  You attempted to read: {inc_filename}.  Verify its existence."
      debug.msg_error(error_msg)
      raise SystemExit("Stop right there!")
    
    if len(inner_df)<1:
        debug.msg_error("There is no content in your data variable.")
        debug.msg_debug("...Verify you copied the input file correctly.")
        raise SystemExit("Stop right there!")
    else:
        debug.msg_debug(f"It appears your data file was read, your data file has {len(inner_df):,} elements of data.")

    return inner_df

IndentationError: expected an indented block after function definition on line 6 (4258641445.py, line 7)

In [ ]:
## Given a Pandas DataFrame, modify the data-types bespoke to this dataset for type consistency
#
#  @param   (inc_dataframe)     - pd.DataFrame  - Dataframe to modify
#  @returns (pd.DataFrame)      - pd.DataFrame  - Modified DataFrame
def convert_datatype_on_dataframe(inc_dataframe: pd.DataFrame) -> pd.DataFrame:
    ####################################################################################################################
    #- Data-type Assignments
    #  Python uses "duck typing" so if it walks like a Duck and quakes like a Duck then...it must be a duck.  Ensure you data is completely consistent for the type.
    ####################################################################################################################
    debug.msg_info(f"Transform columns to data-types you expect. {__name__} {inspect.stack()[0][3]}")
    #the_columns = df.columns
    the_columns = SOURCE_COLUMNS_NAME
    debug.msg_debug(f"...processing: {the_columns}")
    
    for idx, name in enumerate(the_columns):
        try:
            inc_dataframe[name] = inc_dataframe[name].astype(str)
            debug.msg_debug(f"......{name}")
        except Exception as e:
            process_exception(f"Unable to convert {name} to the desired data-type...investigate, continuing:{str(e)}")

    #  ProjectId^LetterId^Text
    #  0         1        2   
    #convert id fields to INTs
    try:
        the_name="ProjectId"
        inc_dataframe[the_name] = inc_dataframe[the_name].astype(int)
        debug.msg_debug(f"......converting {name} to INT")
        the_name="LetterId"
        inc_dataframe[the_name] = inc_dataframe[the_name].astype(int)
        debug.msg_debug(f"......converting {name} to INT")
    except Exception as e:
        process_exception(f"Unable to convert {name} to the desired data-type...investigate, continuing:{str(e)}")


    
    return inc_dataframe

In [ ]:
## Perform data veracity checks (quality controls at the most basic level)
#
#  @param    (inc_dataframe)   - pd.DataFrame  - Dataframe to perform veracity checks on
#  @returns  (pd.DataFrame)    - pd.DataFrame  - Returned dataframe modified to need
def data_varacity_check(inc_dataframe: pd.DataFrame) -> pd.DataFrame:    
    ####################################################################################################################
    #- Rows that don't have sufficient length
    ####################################################################################################################
    debug.msg_info(f"Removal of rows that don't have sufficient string length @ {MINIMUM_LETTER_LENGTH}")
    
    debug.msg_debug("Transform all comments into lower case.")
    #convert all text to lower case first.
    inc_dataframe[SOURCE_COLUMN_NAME] = inc_dataframe[SOURCE_COLUMN_NAME].str.lower()
    
    debug.msg_debug(f"Less than minimal length ({MINIMUM_LETTER_LENGTH}) comments removed.")
    try:
    
        print(f"Failed:{len(inc_dataframe.loc[(inc_dataframe[SOURCE_COLUMN_NAME].str.len() < (MINIMUM_LETTER_LENGTH+1))])}")
        debug.msg_debug(f"......number of records before selecting length > {MINIMUM_LETTER_LENGTH}: {len(inc_dataframe)}")
        df_new = inc_dataframe.loc[(inc_dataframe[SOURCE_COLUMN_NAME].str.len() > MINIMUM_LETTER_LENGTH)]
        df=df_new
        debug.msg_debug(f".......number of records after selecting length > {MINIMUM_LETTER_LENGTH}: {len(df)}")
    except Exception as e:
        process_exception(e)


    ####################################################################################################################
    #- Verbotten Phrases
    #  Are there key phrases in the data that likely shouldn't be there?  Create an array of common phrases that represent non-sensical data, remove them and move on.
    ####################################################################################################################
    debug.msg_info("Verbotten Phrases")
    
    #columns to process (likely only Letter Text)
    the_columns = SOURCE_COLUMNS_NAME
    
    #array of indexes that have phrases that perhaps should be removed
    target_verbotten_rows=[]
    #verbotten_phrases = ['see attach', 'attach',]
    verbotten_phrases = ['see attach']
    
    debug.msg_debug(f"...processing: {the_columns}")
    debug.msg_debug("...removing verbotten phrases.")
    for idx, name in enumerate(the_columns):
        try:
            debug.msg_debug(f"......processing {name}")
            #for index, row in tqdm(df.iterrows(), total=df.shape[0]):
            for idx, row in tqdm(enumerate(df.itertuples(index=False)), total=df.shape[0]):
                for words in verbotten_phrases:
                    if (row[SOURCE_COLUMNS_IDX[0]].find(words)) > 0 and (len(row[SOURCE_COLUMNS_IDX[0]]) < MINIMUM_LETTER_LENGTH):
                        debug.msg_debug("........."+ f'Letter Id:{row[1]}, {words} found at row {idx}.')
                        debug.msg_debug(f"............text says: {row[SOURCE_COLUMNS_IDX[0]][0:200]}")
                        target_verbotten_rows.append(idx)
                        
        #column processing
        except Exception as e:
            process_exception(f"Unable to detect verbotten phrase from {name}...investigate, continuing:{str(e)}")
    
    debug.msg_debug(f"For phrases: {verbotten_phrases}")
    debug.msg_debug(f"   Number of record(s) found: {len(target_verbotten_rows)}.")
    debug.msg_debug(f"   Target Row(s): {target_verbotten_rows}")

    print("\n")
    if len(target_verbotten_rows) > 0:
        debug.msg_debug("Removing target row(s)");
        try:
                debug.msg_debug("Lost Records")
                df_verbotten=df.iloc[target_verbotten_rows]
                debug.msg_debug(f".......number of verbotten records: {len(df_verbotten)}")
        except Exception as e:
                process_exception(f"Unable to to identify df_verbotten...investigate, continuing:{str(e)}")
        print("\n")
            
        try:
                debug.msg_debug("Good Records")
                debug.msg_debug(f"...Number of records before: {len(df)}")
                #df = df.drop(df[<some boolean condition>].index)
                df=df.drop(df_verbotten[:].index)
                debug.msg_debug(f" ...Number of records after: {len(df)}")
        except Exception as e:
                process_exception(f"Unable to remove verbotten row...investigate, continuing:{str(e)}")
        print("\n")

    else:
        debug.msg_debug(f"{BOLD_START}No verbotten phrases detected.{BOLD_END}")
    
    #check for NULLs again
    debug.msg_debug("Columns that have a NULL value.\n")
    print(df.isnull().sum())
    debug.msg_debug("Analyze the result and ensure you have solid data.")
    print("\n")
    df.info()


    ####################################################################################################################
    #- Remove Duplicates known to exist due to data setup
    #  Looking through specific columns are there duplicates of (in this case Author and Text) key columns that are exactly the same?  This is almost guaranteed to be a data anomaly.  Remove the content.
    ####################################################################################################################
    debug.msg_info("Duplicate Removal from Core Fields")
    the_columns=['LetterId', 'Text']
    debug.msg_debug("...removing duplicates due to design of the data.")
    debug.msg_debug(f"......number of records before initial duplicate purge: {len(df)}")
    
    try:
        df = df.drop_duplicates(subset=the_columns)
        
    except Exception as e:
            process_exception(f"Failed to remove duplicates, investigate:{str(e)}")
        
    debug.msg_debug(f".......number of records after initial duplicate purge: {len(df)}")
    
    ####################################################################################################################
    #- Translucent Data
    #  Remove data which could tie back to the user easily, by keeping Id's only and removing personal information, change to a constant, don't remove the field.
    ####################################################################################################################
    debug.msg_info("Translucent Data/PII Filtering\n")
    debug.msg_debug("...no fields in and of themselves to drop.")
    

    ####################################################################################################################
    #- Null Values revealed
    #  Demonstrate where Null fields and analyze the results.
    ####################################################################################################################
    debug.msg_info("Columns that have a NULL value.")
    debug.msg_debug("\n")
    print(df.isnull().sum())
    
    ####################################################################################################################
    #- Populate Null Values (depends on the data)
    #  Fully populate fields, even with stock/default values, where Null values occur.  Create a consistent / rationalized output.
    ####################################################################################################################
    debug.msg_info("Data rationalization, fill the null values.")
    
    try:
        #  ProjectId^LetterId^Text
        #  0         1        2   
        df = df.fillna({'ProjectId': -999, 'LetterId': -999, 'Text': 'Unknown'})
    except Exception as e:
        process_exception(f"Failed to assign values to NULL fields, continuing:{str(e)}")
    
    debug.msg_debug("...columns with NULL values.\n")
    print(df.isnull().sum())
    debug.msg_debug("...analyze the result and ensure you have solid data.")
    
    
    ####################################################################################################################
    #- Drop Records if everything is null
    #  After some cleansing perform another check to ensure no row of data is completly empty, basic cleanup function.
    ####################################################################################################################
    debug.msg_info("Dropping Records Only if All Records are Missing")
    try:
        debug.msg_debug(f"...number of records before dropna: {len(df)}")
        df = df.dropna(how='all')
        debug.msg_debug(f"....number of records after dropna: {len(df)}")
    except Exception as e:
        process_exception(f"Failed to drop rows that are completely blank, continuing:{str(e)}")
    
    ####################################################################################################################
    #- Duplicates
    #  Now that the data is more rationalized ensure you don't have any duplicates.
    ####################################################################################################################
    debug.msg_info("Duplicates Analysis")
    debug.msg_debug(f"...number of records before: {len(df)}")
    if df.duplicated().sum() > 0:
        try:
            # The Pandas .drop_duplicates() method
            df.drop_duplicates(
                subset=None,            # Which columns to consider 
                keep='first',           # Which duplicate record to keep
                inplace=False,          # Whether to drop in place
                ignore_index=False      # Whether to relabel the index
            )
            debug.msg_debug(f"....number of records after: {len(df)}")
        except Exception as e:
            process_exception(f"Failed to remove duplicate values across all rows, continuing:{str(e)}")
    
        # Dropping Based on a Subset of Columns
        #df = df.sort_values(by='Date Modified', ascending=False)
        #df = df.drop_duplicates(subset=['Name', 'Age'], keep='first')
    else:
        debug.msg_debug("No duplicates across all rows found.")

    ####################################################################################################################
    #- Duplicates for a Target set of Rows
    #  For discrete rows, those rows that matter wrt the final solution, ensure no duplicates exist.
    ####################################################################################################################
    debug.msg_info("Remove duplicates of explicit rows.")
    the_columns = SOURCE_COLUMNS_NAME
    debug.msg_debug(f"...processing: {the_columns}")
    
    for column_idx, column_name in enumerate(the_columns):
        debug.msg_debug("...NULL Analysis for "+ str(column_name))
        debug.msg_debug(f"......number of records before: {len(df)}")
        if df[column_name].duplicated().sum() > 0:
            try:
                # The Pandas .drop_duplicates() method
                df.drop_duplicates(
                    subset=column_name,            # Which columns to consider 
                    keep='first',           # Which duplicate record to keep
                    inplace=False,          # Whether to drop in place
                    ignore_index=False      # Whether to relabel the index
                )
                debug.msg_debug(f".......number of records after: {len(df)}")
            except Exception as e:
                process_exception(f"Failed to remove duplicate values across {column_name}, continuing:{str(e)}")
        else:
            debug.msg_debug(f"No duplicates across {column_name} found.")

    
    return df

In [ ]:
## Per string modification of data for cleansing purposes
#  Note: Could be improved to run multi-processing or in batches across the entire payload for greater optimization
#  @param    (inc_text)    - str    - String to modify (cleanse)
#  @param    (run_pii)     - bool   - Do you want to sweep the string for PII and replace PII with constants?
#  @param    (run_cleanse) - bool   - Do you want to sweep the string for non-standard characters and remove them?
#  @param    (run_lemm)    - bool   - Do you want to lemmatize the string?
#  @param    (run_stop)    - bool   - Do you want to remove stop words?
#  @param    (run_stem)    - bool   - Do you want to Stemm the words?
#  @returns  (str)         - str    - Modified content
def data_cleansing(
                   inc_text: str,
                   run_pii: bool,
                   run_cleanse: bool, 
                   run_lemm: bool, 
                   run_stop: bool,
                   run_stem: bool,
                   ) -> str:

    value=inc_text
    if (run_pii):
        #clean the text of PII first since Spacy NER requires complete sentences with lexical relevance
        #removal of special characters can mangle attempts to clean the PII as well
        try:
            #debug.msg_debug("......clean_pii")
            value= clean_pii(value)
        except Exception as e:
            debug.msg_warning("Unable to remove PII from the dataset.")
            process_exception(e)
            
    if (run_cleanse):
        try:
            #debug.msg_debug("......clean_text")
            value= clean_string(value)
        except Exception as e:
            debug.msg_warning("Unable to cleanse the string.")
            process_exception(e)

    if (run_stop):
        try:
            #debug.msg_debug("......clean_stop_words")            
            #note used SpaCy to clean up stop words and it was actually slower, using NLTK
            value= clean_stop_words(value)
        except Exception as e:
            debug.msg_warning("Unable to remove stop words from the string.")
            process_exception(e)

    if (run_lemm):
        try:
            #debug.msg_debug("......clean_lemmatizer_words")            
            #note used SpaCy to lemmatize and it was slower
            value=clean_lemmatizer_words(value)
        except Exception as e:
            debug.msg_warning("Unable to lemmatize the string.")
            process_exception(e)

    if (run_stem):
        try:
            #debug.msg_debug("......clean_stem_words")            
            value=clean_stem_words(value)
        except Exception as e:
            debug.msg_warning("Unable to stem the string.")
            process_exception(e)
    
    #check for NULLs again
    return value



In [ ]:
## Save all work to an MS Excel and Pickled resultant
# 
#  @param    (inc_dataframe)  - pd.DataFrame - DataFrame of content to scan
#  @returns  (None)
def save_results(inc_dataframe: pd.DataFrame,
                 prepended_filename: str):
    
    #CGW, NEW, FIX
    prepended_filename = "TEST"
    
    #establish data version, aligned with code
    data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])

    debug.msg_debug("Create WordCloud.")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001.png"
    create_word_cloud(inc_dataframe, target_filename)

    #save results of all data to binary file
    debug.msg_debug("Saving cleansed/prepped data to binary file.")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001.bin"
    try:
        pickle.dump(inc_dataframe, open(target_filename, "wb"))
        me = pickle.load(open(target_filename, "rb"))
    except (pickle.UnpicklingError, IOError, Exception) as e:
        debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
        process_exception(e)
    debug.msg_debug(f"...saved and reloaded {target_filename}")
    
    #save results of all data to CSV for post-analysis
    debug.msg_debug("Saving cleansed/prepped data to MS Excel file.")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001.csv"
    try:
        inc_dataframe.to_csv(target_filename, 
                    sep='^', 
                    na_rep='', 
                    float_format=None, 
                    columns=None,        #sequence or list of str, optional
                    header=True,         #write out col names, bool
                    index=True, 
                    index_label=None, 
                    errors='strict',
                    storage_options=None, 
                    )
    except (IOError, IllegalCharacterError, Exception) as e:
        debug.msg_warning("FAILED to convert the dataframe to Excel, you might have corruption in the file, an illegal character, or another problem.  Recommend you investigate.")
        process_exception(e)
    
    
    #save results of all data to excel for post-analysis
    debug.msg_debug("Saving cleansed/prepped data to MS Excel file.")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001.xlsx"
    try:
        inc_dataframe.to_excel(target_filename, sheet_name='Comments', 
                    na_rep='', 
                    float_format=None, 
                    columns=None,        #sequence or list of str, optional
                    header=True,         #write out col names, bool
                    index=True, 
                    index_label=None, 
                    startrow=0, 
                    startcol=0, 
                    engine="xlsxwriter", 
                    merge_cells=True, 
                    inf_rep='inf', 
                    freeze_panes=None, 
                    storage_options=None, 
                    )
    except (IOError, IllegalCharacterError, Exception) as e:
        debug.msg_warning("FAILED to convert the dataframe to Excel, you might have corruption in the file, an illegal character, or another problem.  Recommend you investigate.")
        process_exception(e)


In [ ]:
## Perform an inspection of all data for potential prompt injection attacks.  Save results to new fields in the dataframe by direct modification.
#  Formerly: inc_dataframe["PROMPT_INJECTION"]= inc_dataframe[SOURCE_COLUMN_NAME].apply(lambda value: detect_PromptInjection(value))
#  @param    (inc_dataframe)  - pd.DataFrame - DataFrame of content to scan
#  @returns  (None)
def process_prompt_injection(inc_dataframe: pd.DataFrame):
    try:
        the_name='Prompt_Injection'
        the_value='Prompt_Inejection_Content'
        inc_dataframe[the_name]=""
        inc_dataframe[the_name].astype(bool)
        inc_dataframe[the_value]=""
        inc_dataframe[the_value].astype(str)
        #df['PROMPT_INJECTION'][idx] = detect_PromptInjection(df[SOURCE_COLUMN_NAME][idx])
        for idx, row in tqdm(inc_dataframe.iterrows(), total=inc_dataframe.shape[0]):
            #Use final processed text to determine if we have a concern
            resultant, offending_content=detect_PromptInjection(str(row.Text))
            inc_dataframe.loc[idx,the_name]=bool(resultant)
            if resultant:
                inc_dataframe.loc[idx,the_value]="^".join(offending_content)
            else:
                inc_dataframe.loc[idx,the_value]=""
            #print(f"Resultant: {resultant} - {offending_content}")
        debug.msg_debug(f"...prompt injection concerns: {inc_dataframe[the_name].value_counts()}")
    except Exception as e:
        process_exception(e)
    

In [ ]:
def create_word_cloud(inc_dataframe:pd.DataFrame, inc_output_filename: str) -> None:
    ####################################################################################################################
    #- Word Cloud???
    ####################################################################################################################
    debug.msg_info("Word Cloud")
    the_columns = SOURCE_COLUMNS_NAME
    debug.msg_debug(f"...processing: {the_columns}")
    the_data=""
    
    for idx, name in enumerate(the_columns):
        try:
            debug.msg_debug(f"...processing {name} into a single continguous word cloud")
            for index, row in inc_dataframe.iterrows():
                the_data=" ".join([the_data,row[name]])
        except Exception as e:
            process_exception(f"Unable to combine all comments into a contiguous body of text {name}...investigate, continuing:{str(e)}")

        #shouldn't be necessary because this has been cleaned.
        #the_data=str.strip(the_data)
        #the_data.rstrip("\n")
        #the_data=re.sub(r'\W+', ' ', the_data)
        #the_data=re.sub(' +', ' ', the_data)
    
        ########################################
        #Read in the data and perform initial setup
        ########################################
        try:
            text=BeautifulSoup(the_data).get_text()
            cleaned = nltk.word_tokenize(text.lower())
        except Exception as e:
            process_exception(f"Detected trying call BeautifulSoup on the data as follows: {str(e)}")
        ########################################
        #Very basic clean-up continued
        ########################################
        wordlist = [x for x in cleaned if (len(x)>=2 and x.isalpha())]
        
        ########################################
        #API Call (resultant is an image)
        ########################################
        try:
            wordcloud = WordCloud(#stopwords=STOPWORDS,
                                  background_color=IMG_BACKGROUND
                                 ).generate(" ".join(wordlist))
            wordcloud.to_file(inc_output_filename)
            ########################################
            #Show the Results
            ########################################
            plt.figure(figsize=(8.5,11))
            plt.imshow(wordcloud)
            plt.axis('off')
            plt.show()
           
        except Exception as e:
            process_exception(f"Detected trying invoke the WordCloud call as follows: {str(e)}")


In [ ]:
## Read all content from the intended comment and normalize distribution with SpaCy
#
#  @param    (inc_dataframe)  - pd.DataFrame - DataFrame of content to scan
#  @param    (inc_model)      - Spacy Model  - Model loaded to perform analysis of data.
#  @returns  (pd.DataFrame)   - pd.DataFrame - Modified resulting dataframe with comment field normalized.
def normalize_comments_spacy(inc_dataframe: pd.DataFrame, inc_model)-> pd.DataFrame:
    try:
        for idx, row in tqdm(inc_dataframe.iterrows(), total=inc_dataframe.shape[0]):
            new_words=[]
            text_chunks=[]
            if len(row.Text) > inc_model.max_length-1:
                chunks=int(round(len(row.Text)/inc_model.max_length-1))
                start=0
                end=inc_model.max_length-1
                for idx, chunk in enumerate(chunks):
                    start=idx*inc_model.max_length-1
                    end=(idx) * inc_model.max_length-1
                    text_chunks.append(row.Text[start:end])
            else:
                text_chunks.append(row.Text)
                for idx, chunk in enumerate(text_chunks):
                    try:
                        doc=inc_model(chunk)
                        for word in doc:
                            new_words.append(str(word))
                    except Exception as e:
                        debug.msg_warning(f"Normalize comments threw an exception: {repr(e)}")
                        pass  #continue processing regardless, we'll accept loss of some Person identification to keep the code processing
            inc_dataframe.loc[idx,SOURCE_COLUMN_NAME]=" ".join(new_words)
    except Exception as e:
        debug.msg_warning(f"Normalize comments threw an exception: {repr(e)}")
        process_exception(e)
        
    return inc_dataframe

In [ ]:
## Looks for organizational references in the text
#       
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @param (The Model)               - Actual model performing the tokenizing
def detect_organizations(inc_prompt:str, inc_model) -> []:
            
    org_detected=[]
    resultant=set()
    
    try:
        chunk_size=prompt_defense_model_chunk_size
        chunks=prompt_injection_split_string(inc_prompt, chunk_size)
        for chunk_idx, chunk_value in enumerate(chunks):
            doc=inc_model(chunk_value)  
            for token in doc.ents:
                if token.label_ == "ORG":  # Geopolitical Entity (City)
                    org_detected.append(token.text)
    except Exception as e:
        process_exception(f"ERROR finding organizations as follows: {str(e)}")
    
    resultant |= set(org_detected)
    
    return list(resultant)

In [ ]:
## Iterates through text and creates embeddings
#
#  @param (Letter Id as String) - String - Actual Id of comment
#  @param (input text as String) - String - Actual comments that have been cleansed
#. Return JSON payload of embeddings

@backoff.on_exception(backoff.expo, Exception, max_tries=3)
def embed_function(inc_title: str, inc_text:str) -> str:

  import google.generativeai as genai
  resultant=""
  try:
      genai.configure(api_key=os.getenv('GEMINI_USFS_API_KEY'))
      model = EMBEDDING_MODEL_NAME
      resultant = genai.embed_content(model=model,
                                      content=inc_text,
                                      task_type="retrieval_document",
                                      title=inc_title)["embedding"]
       
  except Exception as e:
      #msg_debug.error(f"ERROR detected trying invoke the openai.ChatCompletion.create() call as follows: {str(e)}")
      resultant = {
        "Error": f"{repr(e)}"
       }
  finally:
      return resultant   

In [ ]:
## Looks for location references in the text
#
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @param (The Model)               - Actual model performing the tokenizing
def detect_locations(inc_prompt:str, inc_model) -> []:
        
    loc_detected=[]
    resultant=set()

    try:
        chunk_size=prompt_defense_model_chunk_size
        chunks=prompt_injection_split_string(inc_prompt, chunk_size)
        for chunk_idx, chunk_value in enumerate(chunks):
            doc=inc_model(chunk_value)
            for token in doc.ents:
                if token.label_ in ["LOC", "GPE"]:  # Geopolitical Entity (City)
                    loc_detected.append(token.text)
    except Exception as e:
        process_exception(f"ERROR finding organizations as follows: {str(e)}")

    resultant |= set(loc_detected)
        
    return list(resultant)

In [ ]:
## Looks for address references in the text
#
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @param (The Model)               - Actual model performing the tokenizing
def detect_address(inc_text: str, inc_model) -> str:
    address=""
    zipcode=""
    city=""
    state=""
    resultant=""
    #add_reg=address_pattern.search(inc_text)
    zip_reg=zip_code.search(inc_text)
    #if add_reg:
    #    address = add_reg.group()
    if zip_reg:
        zipcode = zip_reg.group()
        
    doc=inc_model(inc_text)
    for token in doc.ents:
        if token.label_ == "GPE":  # Geopolitical Entity (City)
            city = token.text
        elif token.label_ == "LOC":  # Location (State/Province)
            state = token.text
        elif token.label_ == "DATE":  # Postal Code
            if zipcode == "":
                zipcode = token.text

    try:
        usaddress_values=usaddress.tag(inc_text, tag_mapping={
           'PlaceName': 'city',
           'StateName': 'state',
        })
    except Exception as e:
        debug.msg_warning(f"ADDRESS processing encountered a problem: {str(e)}")
        pass
        #CGW, FIX, TODO, if you have too many identified components the API gets wonky.
        #Need to reduce tokens

    try:
        if city == "":
           city=usaddress_values[0]['city']
    except Exception as e:
        pass
    try:
        if state == "":
            state=usaddress_values[0]['state']
    except Exception as e:
        pass
    try:
        if zipcode == "":
            zipcode=usaddress_values[0]['zip_code']
    except Exception as e:
        pass

    resultant=", ".join([str(city),str(state),str(zipcode)])
    if len(resultant) > 3:
        return resultant
    else:
        return ""

In [ ]:
## Main routine that executes all code, does return a data frame of data for further analysis if desired.
#
#  @param (None)

#turn on profiling to see details about function performance
#@profile_function
def process() -> None:

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")
    
     ####################################################################################################################
    #- Input Source (comments)
    #  New input directly from CARA.
    #  Data format as follows:
    #  ProjectId^LetterId^Text
    #  0         1        2   
    #    Text is original plain text.
    ####################################################################################################################
    debug.msg_info(f"Data Read from Source {__name__} {inspect.stack()[0][3]}")
    debug.msg_debug("...file path declaration")
    
    #year_ranges=["2020",    "2021",   "2022",   "2023",   "2024"]
    #year_formats=["cp1252", "cp1252", "cp1252", "cp1252", "cp1252",]
    year_ranges=["2020"]
    year_formats=["cp1252"]
    
    #CGW, NEW, FIX
    prepended_filename="TEST_"
    
    #read formats
    # cp1252     - windows
    # cp437      - unsure
    # ascii      - standard ascii
    # utf-8      - standard mechanism for common characters today
    # utf-16     - enhanced characters
    # iso-8859-1 - latin encoding
    
    #establish data version, aligned with code
    data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])
    
    #create the core data structure
    df = pd.DataFrame()
    
    debug.msg_debug("Iterating through each year defined in year_ranges and reading each data file into a Pandas DataFrame.")
    for year_idx, year_value in enumerate(year_ranges):
        file_path= DATA_DIR + os.sep + f"CARA_{year_value}_comment.txt"    
        debug.msg_debug(f"...processing {file_path}")            
        loaded_df=load_dataframe(file_path, f"{year_formats[year_idx]}")
        
        #as each dataset is read in inspect the results.
        df = (
              loaded_df
              if df.size == 0
              else pd.concat([df, loaded_df], ignore_index=True)
             )
        debug.msg_debug("......master dataframe updated.")
    
    #CGW, FIX, TODO
    df = df.sample(n=2000)

    ####################################################################################################################
    #- Setup SpaCy model (load)
    ####################################################################################################################
    try:
        spacy.require_gpu()
    except Exception as e:
        #no gpu available will default to all CPU available (this is not considered a problem)
        debug.msg_debug(f"SpaCy GPU registration failed, see exception: {str(e)}")
        pass
    #nlp =spacy.load(SPACY_MODEL, disable=["tok2vec", "tagger", "parser", "attribute_ruler"])
    try:
        nlp =spacy.load(SPACY_MODEL, disable=["tok2vec"])
    except Exception as e:
        debug.msg_warning("Unable to load your model, performing a download instead.")
        #!python -m spacy download {SPACY_MODEL}
        subprocess.run(["python", "-m" , "spacy", "download", SPACY_MODEL])
        pass  #we want to download not cause a problem.
    finally:
        nlp =spacy.load(SPACY_MODEL, disable=["tok2vec"]) 
        
    ####################################################################################################################
    #- Representation of the Data
    #  What does your data actually look like?  Review the table for deeper understanding.
    ####################################################################################################################
    print(df.info())
    #print(df.describe())
    df=convert_datatype_on_dataframe(df)

    ####################################################################################################################
    #- Backup Original Data
    ####################################################################################################################
    debug.msg_info("Backup transformation column.")
    df[SOURCE_COLUMN_NAME + "_Original"]=df[SOURCE_COLUMN_NAME]
    
    ####################################################################################################################
    #- Duplicate removal, etc.
    ####################################################################################################################
    debug.msg_info("Data Varacity Check")
    df=data_varacity_check(df)

    ####################################################################################################################
    #- Consistent "mooshing" of last part of comment, especially with "sincerely" fix
    ####################################################################################################################
    debug.msg_info("Fixing Sincerely problem and parenthetical problem.")    
    try:
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            repaired_string=re.sub('.sincerely',". sincerely",row.Text, re.IGNORECASE)
            repaired_string=re.sub('.respectfully',". respectfully",repaired_string, re.IGNORECASE)
            repaired_string=re.sub('\)[A-Z,a-z]',") ",repaired_string, re.IGNORECASE)
            df.loc[idx,SOURCE_COLUMN_NAME]=repaired_string
            #print(f"Processing: {df.loc[idx,SOURCE_COLUMN_NAME]}")
            #print()
            #break            
    except Exception as e:
        process_exception(e)
    finally:
        df[SOURCE_COLUMN_NAME].astype(str)        

    ####################################################################################################################
    #- ORG harvesting
    ####################################################################################################################
    debug.msg_info("Organization Harvest, saving to separate file")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001_ORGS.csv"
    try:
        with open(target_filename, "w") as my_file:
            my_file.write("LetterId^Orgs\n")                        
            for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
                the_orgs=detect_organizations(row.Text,nlp)
                #df.loc[idx,target_column_name]=detect_organizations(row.Text, nlp)
                for idx,name in enumerate(the_orgs):
                    my_file.write(f"{row.LetterId}^{name.encode('utf-8')}\n")
    except (IOError, Exception) as e:
        debug.msg_warning("FAILED to open and write to a file for ORGS, you might have corruption, investigate.")
        process_exception(e)
    ####################################################################################################################
    #- LOCATION harvesting
    ####################################################################################################################
    debug.msg_info("Location Harvest")
    #df.loc[idx,target_column_name]==detect_locations(row.Text, nlp)
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001_LOCS.csv"
    try:
        with open(target_filename, "w") as my_file:
            my_file.write("LetterId^Locations\n")                        
            for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
                the_locs=detect_locations(row.Text, nlp)
                for idx,name in enumerate(the_locs):
                    my_file.write(f"{row.LetterId}^{name.encode('utf-8')}\n")
    except (IOError, Exception) as e:
        debug.msg_warning("FAILED to open and write to a file for LOCATIONS, you might have corruption, investigate.")
        process_exception(e)

    ####################################################################################################################
    #- Geospatial harvesting
    ####################################################################################################################
    debug.msg_info("Address Harvest")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001_ADDRESS.csv"
    try:
        with open(target_filename, "w") as my_file:
            my_file.write("LetterId^Address\n")                        
            for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
                the_answer=detect_address(row.Text, nlp)
                if the_answer != "":
                    my_file.write(f"{row.LetterId}^{the_answer.encode('utf-8')}\n")
    except (IOError, Exception) as e:
        debug.msg_warning("FAILED to open and write to a file for ADDRESS, you might have corruption, investigate.")
        process_exception(e)
        
        
        
    ####################################################################################################################
    #- Basic Metrics
    ####################################################################################################################
    debug.msg_info("Processing initial metrics.")
    get_metrics(df)

    ####################################################################################################################
    #- First PII Clean-up
    ####################################################################################################################
    debug.msg_info("Cleaning Pii from text (1st Sweep).")
    try:
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            df.loc[idx,SOURCE_COLUMN_NAME]=data_cleansing(str(row.Text), True, False, False, False, False)
            #print(f"Processing: {df.loc[idx,SOURCE_COLUMN_NAME]}")
            #print()
            #break
    except Exception as e:
        process_exception(e)
    finally:
        df[SOURCE_COLUMN_NAME].astype(str)    
    
    ####################################################################################################################
    #- REMOVE PEOPLE NAMES
    ####################################################################################################################
    debug.msg_info("Removing named entities (PERSON) from text.")
    try:
        #df[SOURCE_COLUMN_NAME]= df[SOURCE_COLUMN_NAME].apply(lambda value: clean_named_entity_recognition(nlp,value))
        #want to see progress as apply is simply a black box for data this large
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            df.loc[idx,SOURCE_COLUMN_NAME]=clean_named_entity_recognition(nlp, row.Text)
            #print(f"Processing: {df.loc[idx,SOURCE_COLUMN_NAME]}")
            #print()
            #break
    except Exception as e:
        process_exception(e)
    finally:
        df[SOURCE_COLUMN_NAME].astype(str)

    ####################################################################################################################
    #- REMOVE EMAIL
    ####################################################################################################################
    debug.msg_info("Removing email from text.")
    matcher = Matcher(nlp.vocab)
    pattern = [{"LIKE_EMAIL": True}]
    matcher.add("EMAIL_ADDRESS", [pattern])
    try:
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            df.loc[idx,SOURCE_COLUMN_NAME]=clean_email_spacy(nlp, matcher, row.Text)
            #print(f"Processing: {df.loc[idx,SOURCE_COLUMN_NAME]}")
            #print()
            #break
    except Exception as e:
        process_exception(e)
    finally:
        df[SOURCE_COLUMN_NAME].astype(str)
    
    ####################################################################################################################
    #- CLEAN TEXT OF PII and extra characters (2nd sweep of PII)
    ####################################################################################################################
    debug.msg_info("Cleaning text.")
    try:
        #df[SOURCE_COLUMN_NAME][idx] = data_cleansing(df[SOURCE_COLUMN_NAME][idx], True, True, False, False, False)
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            df.loc[idx,SOURCE_COLUMN_NAME]=data_cleansing(str(row.Text), True, True, False, False, False)
            #print(f"Processing: {df.loc[idx,SOURCE_COLUMN_NAME]}")
            #print()
            #break
    except Exception as e:
        process_exception(e)
    finally:
        df[SOURCE_COLUMN_NAME].astype(str)

    ####################################################################################################################
    #- EMBEDDINGS
    ####################################################################################################################
    debug.msg_info("Create embeddings to a separate file")
    target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments_001_EMBEDDINGS.csv"
    try:
        with open(target_filename, "w") as my_file:
            my_file.write("LetterId^Embedding\n")            
            for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
                my_file.write(f"{row.LetterId}^{embed_function(str(row.LetterId),str(row.Text))}\n")
    except (IOError, Exception) as e:
        debug.msg_warning("FAILED to open and write to a file for EMBEDDINGS, you might have corruption, investigate.")
        process_exception(e)        
        
    ####################################################################################################################
    #- LEMMATIZE TEXT (save to new column for analysis comparison)
    ####################################################################################################################
    debug.msg_info("Lemmatizing text.")
    try:
        the_name="Text_LemmStop"
        df[the_name]=""
        df[the_name].astype(str)
        for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
            df.loc[idx,the_name]=data_cleansing(str(row.Text), False, False, True, True, False)
            #print(f"Processing: {df.loc[idx,the_name]}")
            #print()
            #break
    except Exception as e:
        process_exception(e)
    finally:
        df[the_name].astype(str)
        
    ####################################################################################################################
    #- Prompt Injection Defense (creates new true/false attribute with potential identification of "true" results)
    ####################################################################################################################
    debug.msg_info("Prompt Injection Defense")
    process_prompt_injection(df)

    ####################################################################################################################
    #- Basic Metrics again now that the data has been altered
    ####################################################################################################################
    print("Metrics after stop words are applied.")
    get_metrics(df)
    
    ####################################################################################################################
    #- Cleanse all data to utf-8 removing illegal characters
    ####################################################################################################################
    print("UTF-8 Transformation")
    df = df.applymap(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x,str) else x)
    
    ####################################################################################################################
    #- Save results to datafile (MS Excel and Pickled file for later re-use in 002)
    ####################################################################################################################
    debug.msg_info("Saving completed data file")
    save_results(df, prepended_filename)
   
    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")

## Main 

In [ ]:
if __name__ == "__main__":

    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")
    
    
    #with warnings.catch_warnings():
    # To ignore specific warning types:
    warnings.filterwarnings('ignore', category=DeprecationWarning)
    warnings.filterwarnings('ignore', category=FutureWarning)
    warnings.filterwarnings('ignore', category=UserWarning)

    # Code that generates the deprecation warning

    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    os.environ['PYTHONIOENCODING']=ENCODING
    #spacy requirement
    os.environ['TOKENIZERS_PARALLELISM']="false"
    
    ############################################
    # GLOBAL VARIABLES
    ############################################
    debug.msg_info("Variable declaration.")    
    DEBUG = 1
    DEBUG_DATA = 0
    
    # CODE CONSTRAINTS
    VERSION_NAME    = "CMTANL"
    VERSION_MAJOR   = 0
    VERSION_MINOR   = 7
    VERSION_RELEASE = 0
    
    TEXT_WIDTH=77
    BOLD_START = "\033[1m"
    BOLD_END = "\033[0;0m"
    
    ###########################################
    #- API Parameters for things like WordCloud
    ###########################################
    IMG_BACKGROUND=None                        #None without quotes or "black", "white", etc...
    IMG_FONT_SIZE_MIN=14
    IMG_WIDTH=800
    IMG_HEIGHT=600
    
    ############################################
    # SECRETS and ENV VARIABLES
    ############################################
    load_dotenv()  # take environment variables

    ############################################
    # APPLICATION VARIABLES
    ############################################
    DELIM='^'
    SPELL_CHECK_DISTANCE=2
    MINIMUM_AI_RESPONSE=25                     #words
    MINIMUM_AI_WAIT=15                         #seconds
    os.environ["MINIMUM_AI_WAIT"] = "str(MINIMUM_AI_WAIT)"
    MINIMUM_LETTER_LENGTH=15
    
    # CARA_2020_comment.txt
    #  ProjectId^LetterId^Text
    #  0         1        2   
    SOURCE_COLUMNS_NAME=["Text"]        #body of text where the actual comment is
    SOURCE_COLUMNS_IDX=[ 2 ]                   #location in data frame AFTER removal of columns
    SOURCE_COLUMN_NAME=SOURCE_COLUMNS_NAME[0]
    
    EVALUATION_RECORDS=10
    ERROR_PHRASE = 'Error code: 400'
    OPENAI_RESULT="ResultOPENAI"
    DATA_DIR='/home/jupyter/projects/data/source_data/nlp/EMC_PUBLIC_COMMENTS_RAW/'

    EMBEDDING_MODEL_NAME='models/embedding-001'
    SPACY_MODEL         ="en_core_web_trf"       #[en, en_core_web_sm, en_core_web_lg, en_core_web_trf]
    PROMPT_INJECTION_MODELS=[
                             #https://python.langchain.com/v0.1/docs/guides/productionization/safety/hugging_face_prompt_injection/
                             #from optimum.onnxruntime import ORTModelForSequenceClassification
                             "protectai/deberta-v3-base-prompt-injection-v2",
                            ]

    ########################################
    #Geospatial Patterns and Data
    ########################################
    address_pattern = re.compile('\d{1,4} [\w\s]{1,20}(?:street|st|avenue|ave|road|rd|highway|hwy|square|sq|trail|trl|drive|dr|court|ct|park|parkway|pkwy|circle|cir|boulevard|blvd)\W?(?=\s|$)', re.IGNORECASE)
    zip_code = re.compile(r'\b\d{5}(?:[-\s]\d{4})?\b')
    #states = ["AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DC", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"]

    ########################################
    #Define Potential "answers" from the various neural layers
    #Is the Prompt Detected?
    ########################################
    id2label = {
        'LEGIT':    False,
        'POSITIVE': False,
        'LABEL_1':  False,
        'SAFE':     False,
        
        'INJECTION':True,
        'NEGATIVE': True,
        'LABEL_0':  True,
        'UNSAFE':   True,
    }
    prompt_defense_model_chunk_size=512  

    ####################################################################################################################
    #- Invocation of functions and instantiation of system needs, nltk instantiation
    ####################################################################################################################
    debug.msg_debug(f"...StopWords instantiated.")
    stop_words = set(stopwords.words("english"))
    debug.msg_debug(f"...PortStemmer instantiated.")
    stemmer = PorterStemmer()
    debug.msg_debug(f"...Lemmatizer instantiated.")
    lemmatizer = WordNetLemmatizer()
    
    #setup the text wrapper
    debug.msg_debug(f"...Text Wrapper instantiated.")
    wrapper = textwrap.TextWrapper(width=TEXT_WIDTH)
    
    #show your libraries
    lib_diagnostics()
    
    #release memory from the GPU
    from numba import cuda 
    try:
        device = cuda.get_current_device()
        device.reset()
    except Exception as e:
        process_exception(f"ERROR releasing memory from the GPU, see: {str(repr(e))}")
    
    #load the injection models for future use
    the_models=[]
    try:
        debug.msg_info(f"Loading prompt injection model(s)")
        for model_idx,model_name in enumerate(PROMPT_INJECTION_MODELS):
            the_models.append(pipeline("text-classification", model=str(model_name), device=0))
            debug.msg_debug(f"...loaded {model_name}")
    except Exception as e:
        process_exception(f"ERROR detected trying to use the transformer.pipeline API to load hugging face models as follows: {str(e)}")
        
    process()
    
    end_t=perf_counter()
    print("END PROGRAM")
    print(f"Elapsed time: {end_t - start_t}")    